<a href="https://colab.research.google.com/github/xiyuan1avery/ma2288/blob/main/notebooks/03_prepare_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prepare Dataset and Extract Teacher Hidden States

This notebook:

1. downloads WikiText-2;
2. tokenizes the text into fixed-length sequences;
3. extracts frozen DistilGPT-2 hidden states;
4. saves the resulting tensors to Google Drive.

In [1]:
!pip install -q transformers datasets accelerate

In [2]:
import gc
import random
from pathlib import Path

import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model = model.to(device)
model.eval()

# 冻结全部语言模型参数
for parameter in model.parameters():
    parameter.requires_grad = False

print("Model:", MODEL_NAME)
print("Hidden dimension:", model.config.n_embd)
print("Number of layers:", model.config.n_layer)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: distilgpt2
Hidden dimension: 768
Number of layers: 6


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
)

print(dataset)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [5]:
for split_name in ["train", "validation", "test"]:
    print(f"\n--- {split_name.upper()} EXAMPLES ---")

    shown = 0

    for example in dataset[split_name]:
        text = example["text"].strip()

        if text:
            print(repr(text[:150]))
            shown += 1

        if shown == 3:
            break


--- TRAIN EXAMPLES ---
'= Valkyria Chronicles III ='
'Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chroni'
'The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard feature'

--- VALIDATION EXAMPLES ---
'= Homarus gammarus ='
'Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Se'
'= = Description = ='

--- TEST EXAMPLES ---
'= Robert Boulter ='
'Robert Boulter is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This w'
'In 2006 , Boulter starred alongside Whishaw in the play Citizenship written by Mark Ravenhill . He appeared on a 2006 episode of the television series'


In [6]:
SEQUENCE_LENGTH = 64

MAX_BLOCKS = {
    "train": 400,
    "validation": 80,
    "test": 80,
}


def make_token_blocks(dataset_split, max_blocks):
    """
    Convert a WikiText split into fixed-length token sequences.

    Output shape:
        [number_of_blocks, SEQUENCE_LENGTH]
    """

    # 删除空白文本
    nonempty_texts = [
        example["text"].strip()
        for example in dataset_split
        if example["text"].strip()
    ]

    # 用 GPT-2 的 end-of-text token 分隔不同文本段
    separator = tokenizer.eos_token
    combined_text = separator.join(nonempty_texts)

    # 把文本转换成 token IDs
    token_ids = tokenizer(
        combined_text,
        add_special_tokens=False,
        return_attention_mask=False,
    )["input_ids"]

    required_tokens = max_blocks * SEQUENCE_LENGTH
    token_ids = token_ids[:required_tokens]

    # 丢弃不足一个完整 block 的最后部分
    usable_length = (
        len(token_ids) // SEQUENCE_LENGTH
    ) * SEQUENCE_LENGTH

    token_ids = token_ids[:usable_length]

    # 转换成 PyTorch tensor
    token_tensor = torch.tensor(
        token_ids,
        dtype=torch.long,
    )

    # 从一条长序列变成多个长度为 64 的序列
    token_blocks = token_tensor.reshape(
        -1,
        SEQUENCE_LENGTH,
    )

    return token_blocks

In [7]:
token_blocks = {}

for split_name in ["train", "validation", "test"]:
    token_blocks[split_name] = make_token_blocks(
        dataset[split_name],
        MAX_BLOCKS[split_name],
    )

    print(
        split_name,
        "shape:",
        token_blocks[split_name].shape,
    )

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2370804 > 1024). Running this sequence through the model will result in indexing errors


train shape: torch.Size([400, 64])
validation shape: torch.Size([80, 64])
test shape: torch.Size([80, 64])


In [8]:
example_block = token_blocks["train"][0]

print("Token IDs:")
print(example_block.tolist())

print("\nDecoded text:")
print(tokenizer.decode(example_block))

Token IDs:
[28, 569, 18354, 7496, 17740, 6711, 796, 50256, 10445, 73, 13090, 645, 569, 18354, 7496, 513, 1058, 791, 47398, 17740, 357, 4960, 1058, 10545, 230, 99, 161, 254, 112, 5641, 44444, 9202, 25084, 24440, 12675, 11839, 18, 837, 6578, 764, 569, 18354, 7496, 286, 262, 30193, 513, 1267, 837, 8811, 6412, 284, 355, 569, 18354, 7496, 17740, 6711, 2354, 2869, 837, 318, 257, 16106]

Decoded text:
= Valkyria Chronicles III =<|endoftext|>Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical


In [9]:
BATCH_SIZE = 8


def extract_hidden_states(token_tensor, split_name):
    """
    Run frozen DistilGPT-2 and extract final-layer hidden states.

    Input shape:
        [number_of_blocks, sequence_length]

    Output shape:
        [number_of_blocks, sequence_length, hidden_dimension]
    """

    hidden_batches = []

    total_blocks = token_tensor.shape[0]

    for start in range(0, total_blocks, BATCH_SIZE):
        end = min(start + BATCH_SIZE, total_blocks)

        batch_input_ids = token_tensor[start:end].to(device)

        # 所有序列长度相同，因此不需要 padding
        batch_attention_mask = torch.ones_like(
            batch_input_ids,
            device=device,
        )

        with torch.no_grad():
            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                output_hidden_states=True,
                return_dict=True,
            )

        final_hidden = outputs.hidden_states[-1]

        # 转换成 float16 并移回 CPU，减少内存占用
        final_hidden = final_hidden.to(
            dtype=torch.float16,
            device="cpu",
        )

        hidden_batches.append(final_hidden)

        if start == 0 or end == total_blocks or end % 80 == 0:
            print(
                f"{split_name}: processed "
                f"{end}/{total_blocks} blocks"
            )

    return torch.cat(hidden_batches, dim=0)

In [10]:
hidden_states = {}

for split_name in ["train", "validation", "test"]:
    print(f"\nExtracting {split_name} hidden states...")

    hidden_states[split_name] = extract_hidden_states(
        token_blocks[split_name],
        split_name,
    )

    print(
        split_name,
        "hidden-state shape:",
        hidden_states[split_name].shape,
    )


Extracting train hidden states...
train: processed 8/400 blocks
train: processed 80/400 blocks
train: processed 160/400 blocks
train: processed 240/400 blocks
train: processed 320/400 blocks
train: processed 400/400 blocks
train hidden-state shape: torch.Size([400, 64, 768])

Extracting validation hidden states...
validation: processed 8/80 blocks
validation: processed 80/80 blocks
validation hidden-state shape: torch.Size([80, 64, 768])

Extracting test hidden states...
test: processed 8/80 blocks
test: processed 80/80 blocks
test hidden-state shape: torch.Size([80, 64, 768])


In [11]:
for split_name in ["train", "validation", "test"]:
    tokens = token_blocks[split_name]
    states = hidden_states[split_name]

    print(f"\n{split_name}:")
    print("  Token shape:", tokens.shape)
    print("  Hidden shape:", states.shape)
    print("  Token dtype:", tokens.dtype)
    print("  Hidden dtype:", states.dtype)
    print("  Contains NaN:", torch.isnan(states).any().item())
    print("  Contains infinity:", torch.isinf(states).any().item())


train:
  Token shape: torch.Size([400, 64])
  Hidden shape: torch.Size([400, 64, 768])
  Token dtype: torch.int64
  Hidden dtype: torch.float16
  Contains NaN: False
  Contains infinity: False

validation:
  Token shape: torch.Size([80, 64])
  Hidden shape: torch.Size([80, 64, 768])
  Token dtype: torch.int64
  Hidden dtype: torch.float16
  Contains NaN: False
  Contains infinity: False

test:
  Token shape: torch.Size([80, 64])
  Hidden shape: torch.Size([80, 64, 768])
  Token dtype: torch.int64
  Hidden dtype: torch.float16
  Contains NaN: False
  Contains infinity: False


In [12]:
block_index = 0
position = 10

example_tokens = token_blocks["train"][block_index]
example_states = hidden_states["train"][block_index]

h_t = example_states[position]
next_token_id = example_tokens[position + 1]
h_t_plus_1 = example_states[position + 1]

print("Current position:", position)
print("Current token:", repr(tokenizer.decode([example_tokens[position].item()])))
print("Next token:", repr(tokenizer.decode([next_token_id.item()])))
print("h_t shape:", h_t.shape)
print("h_t+1 shape:", h_t_plus_1.shape)

Current position: 10
Current token: 'ō'
Next token: ' no'
h_t shape: torch.Size([768])
h_t+1 shape: torch.Size([768])


In [13]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [14]:
DRIVE_DIRECTORY = Path(
    "/content/drive/MyDrive/ma2288_nextlat"
)

DRIVE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

print("Saving to:", DRIVE_DIRECTORY)

Saving to: /content/drive/MyDrive/ma2288_nextlat


In [15]:
for split_name in ["train", "validation", "test"]:
    output_path = (
        DRIVE_DIRECTORY
        / f"{split_name}_teacher_states.pt"
    )

    artifact = {
        "model_name": MODEL_NAME,
        "sequence_length": SEQUENCE_LENGTH,
        "token_ids": token_blocks[split_name],
        "hidden_states": hidden_states[split_name],
    }

    torch.save(artifact, output_path)

    file_size_mb = output_path.stat().st_size / 1024**2

    print(
        f"Saved {split_name} to {output_path}"
    )
    print(
        f"File size: {file_size_mb:.2f} MB"
    )

Saved train to /content/drive/MyDrive/ma2288_nextlat/train_teacher_states.pt
File size: 37.70 MB
Saved validation to /content/drive/MyDrive/ma2288_nextlat/validation_teacher_states.pt
File size: 7.54 MB
Saved test to /content/drive/MyDrive/ma2288_nextlat/test_teacher_states.pt
File size: 7.54 MB


In [16]:
test_path = (
    DRIVE_DIRECTORY
    / "test_teacher_states.pt"
)

loaded_test = torch.load(
    test_path,
    map_location="cpu",
    weights_only=False,
)

print("Loaded model:", loaded_test["model_name"])
print("Loaded sequence length:", loaded_test["sequence_length"])
print("Loaded token shape:", loaded_test["token_ids"].shape)
print("Loaded hidden shape:", loaded_test["hidden_states"].shape)

Loaded model: distilgpt2
Loaded sequence length: 64
Loaded token shape: torch.Size([80, 64])
Loaded hidden shape: torch.Size([80, 64, 768])
